In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score


In [446]:
# The notebook kernel may start outside the project folder.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates += [folder / "applied-ml-midterm" for folder in candidates.copy()]
# Local project location, used if the kernel starts in an unrelated folder.
candidates.append(Path("/Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm"))

PROJECT_DIR = next((folder for folder in candidates if (folder / "part1-data").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Could not find part1-data. Kernel working directory: {cwd}")

ATTEMPT_DIR = PROJECT_DIR / "prith_attempts" / "part_1"
DATA_DIR = PROJECT_DIR / "part1-data"
print("Loading data from:", DATA_DIR)

part1_train = pd.read_csv(DATA_DIR / "part1_train.csv")
part1_val = pd.read_csv(DATA_DIR / "part1_val.csv")
part1_test = pd.read_csv(DATA_DIR / "part1_test.csv")
pathway_catalog = pd.read_csv(DATA_DIR / "pathway_catalog.csv")

print("Training:", part1_train.shape)
print("Validation:", part1_val.shape)
print("Test:", part1_test.shape)
print("Pathway catalog:", pathway_catalog.shape)

part1_train.head()


Loading data from: /Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm/part1-data
Training: (1171, 166)
Validation: (78, 166)
Test: (312, 161)
Pathway catalog: (2501, 5)


,ID,start_year,criteria,diseases_cleaned,drugs,icdcodes,matched_drugs,drug_targets,drug_actions,drug_pathways,...,emb_145,emb_146,emb_147,emb_148,emb_149,Safety,Funding,Efficacy,Recruitment,Other
0,TR08860,2003,\n Inclusion Criteria:\n\n - ...,"['lung cancer, non-small cell']",['gw572016 (lapatinib)'],"[""['C78.00', 'C78.01', 'C78.02', 'D14.30', 'D1...",NaN,NaN,NaN,NaN,...,0.057633,-0.018126,-0.130373,-0.072231,-0.051629,-100,-100,-100,-100,-100
1,TR03494,2004,\n DISEASE CHARACTERISTICS:\n\n ...,['leukemia'],"['autologous tumor cell vaccine', 'therapeutic...","[""['C95.91', 'C95.92', 'Z80.6', 'Z85.6', 'C90....",NaN,NaN,NaN,NaN,...,-0.018860,-0.077071,-0.059479,0.009044,0.021266,-100,-100,-100,-100,-100
2,TR00772,2006,\n Inclusion Criteria:\n\n - ...,"['clear cell renal cell carcinoma', 'recurrent...",['cediranib maleate'],"[""['C22.0', 'C4A.9', 'C7B.1', 'C4A.0', 'C4A.31...",NaN,NaN,NaN,NaN,...,-0.019414,-0.007497,-0.062808,-0.071619,-0.078172,-100,-100,-100,-100,-100
3,TR08870,2007,\n Inclusion Criteria:\n\n - ...,"['pulmonary disease, chronic obstructive']","['uk-432,097', 'placebo']","[""['J44.9', 'J44.1', 'J44.0']""]",NaN,NaN,NaN,NaN,...,0.014069,-0.006230,-0.081606,-0.031338,-0.063282,-100,-100,-100,-100,-100
4,TR02827,2004,\n Inclusion Criteria:\n\n - ...,"['schizophrenia', 'schizoaffective disorder']",['ziprasidone'],"[""['F20.0', 'F20.1', 'F20.2', 'F20.3', 'F20.5'...",ziprasidone,ADRA1A;ADRA1B;ADRA2A;ADRA2B;ADRA2C;CHRM1;CHRM2...,agonist;antagonist;binder,R-HSA-109582;R-HSA-112314;R-HSA-112315;R-HSA-1...,...,0.093187,-0.043244,-0.077393,-0.110769,0.016161,-100,-100,-100,-100,-100


In [447]:
def create_scaled_embeddings(train_df, val_df, test_df):
    embedding_columns = [f"emb_{i}" for i in range(150)]

    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[embedding_columns])
    X_val = scaler.transform(val_df[embedding_columns])
    X_test = scaler.transform(test_df[embedding_columns])

    return X_train, X_val, X_test, scaler



In [448]:
def train_kmeans(X_train, n_clusters, n_init=100, random_state=42):
    model = KMeans(
        n_clusters=n_clusters,
        n_init=n_init,
        random_state=random_state,
    )
    model.fit(X_train)
    return model


In [ ]:
def inspect_cluster_percentages(model, X_train, train_df, percentage_threshold):
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    cluster_ids = model.predict(X_train)
    labeled_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()

    clusters = range(model.n_clusters)
    cluster_reason_mapping = {}
    for cluster in clusters:
        labeled_positions = np.flatnonzero((cluster_ids == cluster) & labeled_mask)
        cluster_reason_mapping[int(cluster)] = []
        if len(labeled_positions) == 0:
            continue

        labeled_rows = train_df.iloc[labeled_positions]
        percentages = labeled_rows[reason_columns].mean() * 100
        cluster_reason_mapping[int(cluster)] = [
            reason for reason in reason_columns
            if percentages[reason] >= percentage_threshold
        ]

    return cluster_reason_mapping


In [450]:
def label_unlabeled_points(model, X_train, train_df, cluster_reason_mapping):
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    known_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()
    unlabeled_mask = train_df[reason_columns].eq(-100).all(axis=1).to_numpy()

    result = train_df.copy()
    result["label_source"] = np.where(known_mask, "observed", "unresolved").astype(object)
    cluster_ids = model.predict(X_train)
    reason_positions = result.columns.get_indexer(reason_columns)

    for position in np.flatnonzero(unlabeled_mask):
        reasons = cluster_reason_mapping.get(int(cluster_ids[position]), [])
        if not reasons:
            continue
        result.iloc[position, reason_positions] = [
            int(reason in reasons) for reason in reason_columns
        ]
        result.iloc[position, result.columns.get_loc("label_source")] = "cluster_prediction"

    return result


In [451]:
def train_logistic_regression(X_train, train_df, C=1.0, max_iter=500,
                              random_state=42):
    """Train five classifiers using observed and recovered labels equally."""
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    labeled_mask = train_df[reason_columns].isin([0, 1]).all(axis=1).to_numpy()
    if not labeled_mask.any():
        raise ValueError("No rows have five known binary reason labels for training.")

    y_train = train_df.loc[labeled_mask, reason_columns].to_numpy(dtype=int)
    model = OneVsRestClassifier(
        LogisticRegression(
            C=C,
            max_iter=max_iter,
            solver="liblinear",
            random_state=random_state,
        )
    )
    model.fit(X_train[labeled_mask], y_train)
    return model


In [452]:
def predict_reason_labels(model, X):
    """Predict binary reasons, choosing the highest-probability reason if all are zero."""
    predictions = np.asarray(model.predict(X)).copy()
    empty_rows = ~predictions.any(axis=1)
    if empty_rows.any():
        probabilities = model.predict_proba(X[empty_rows])
        best_reasons = np.asarray(probabilities).argmax(axis=1)
        predictions[np.flatnonzero(empty_rows), best_reasons] = 1
    return predictions


In [453]:
def evaluate_reason_model(model, X_val, val_df):
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    y_true = val_df[reason_columns].to_numpy(dtype=int)
    y_pred = predict_reason_labels(model, X_val)
    per_reason_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

    return pd.Series(
        [
            f1_score(y_true, y_pred, average="samples", zero_division=0),
            f1_score(y_true, y_pred, average="macro", zero_division=0),
            *per_reason_f1,
        ],
        index=["Sample-wise F1", "Macro F1", *reason_columns],
        name="Validation F1",
    )


In [454]:
def create_pca_embeddings(X_train, n_components=50):
    """Fit PCA on scaled training embeddings and return embeddings plus fitted PCA."""
    pca = PCA(n_components=n_components, svd_solver="full")
    X_train_pca = pca.fit_transform(X_train)
    return X_train_pca, pca


In [455]:
def create_kernel_pca_embeddings(X_train, n_components=50, kernel="rbf",
                                 gamma=None, random_state=42):
    """Fit kernel PCA on scaled training embeddings; return embeddings and transformer."""
    effective_gamma = 1 / X_train.shape[1] if gamma is None else gamma
    kernel_pca = KernelPCA(
        n_components=n_components,
        kernel=kernel,
        gamma=effective_gamma,
        eigen_solver="arpack",
        random_state=random_state,
    )
    X_train_kernel_pca = kernel_pca.fit_transform(X_train)
    return X_train_kernel_pca, kernel_pca


In [ ]:
# Run the imports, data-loading, and function-definition cells first.
PERCENTAGE_THRESHOLD = 50
RANDOM_STATE = 42  # Kernel PCA and logistic regression stay fixed.
KMEANS_RANDOM_STATE = 42
N_INIT = 100
KERNEL_PCA_COMPONENTS = 50
KERNEL = "rbf"
KERNEL_GAMMA = 1 / 75

# Scaling is shared across runs and fitted on training embeddings only.
X_train, X_val, X_test, scaler = create_scaled_embeddings(
    part1_train, part1_val, part1_test
)

# Kernel PCA is used only for clustering and label recovery.
X_train_cluster, kernel_pca_model = create_kernel_pca_embeddings(
    X_train, n_components=KERNEL_PCA_COMPONENTS, kernel=KERNEL,
    gamma=KERNEL_GAMMA, random_state=RANDOM_STATE,
)
print(f"Clustering uses {X_train_cluster.shape[1]} kernel PCA components.")
print(f"Kernel: {kernel_pca_model.kernel}; gamma: {kernel_pca_model.gamma:.6g}")

# The baseline and augmented classifiers both use the original scaled embeddings.
baseline_model = train_logistic_regression(
    X_train, part1_train, C=1.0, max_iter=500, random_state=RANDOM_STATE,
)
baseline_validation_metrics = evaluate_reason_model(baseline_model, X_val, part1_val)

pipeline_results = {}
# Keep the selected K=21 configuration fixed.
# Use range(2, 31) here to restore a cluster-count sweep later.
for n_clusters in [21]:
    kmeans_model = train_kmeans(
        X_train_cluster, n_clusters=n_clusters,
        n_init=N_INIT, random_state=KMEANS_RANDOM_STATE,
    )
    cluster_reason_mapping = inspect_cluster_percentages(
        kmeans_model, X_train_cluster, part1_train,
        percentage_threshold=PERCENTAGE_THRESHOLD,
    )
    part1_train_augmented = label_unlabeled_points(
        kmeans_model, X_train_cluster, part1_train, cluster_reason_mapping
    )
    reason_model = train_logistic_regression(
        X_train, part1_train_augmented,
        C=1.0, max_iter=500, random_state=RANDOM_STATE,
    )
    validation_metrics = evaluate_reason_model(reason_model, X_val, part1_val)

    pipeline_results[n_clusters] = {
        "kmeans_random_state": KMEANS_RANDOM_STATE,
        "n_init": N_INIT,
        "kernel_pca_model": kernel_pca_model,
        "kmeans_model": kmeans_model,
        "cluster_reason_mapping": cluster_reason_mapping,
        "part1_train_augmented": part1_train_augmented,
        "reason_model": reason_model,
        "validation_metrics": validation_metrics,
        "label_source_counts": part1_train_augmented["label_source"].value_counts(),
    }


In [ ]:
validation_scores = pd.DataFrame.from_dict(
    {
        n_clusters: run["validation_metrics"]
        for n_clusters, run in pipeline_results.items()
    },
    orient="index",
).sort_index()
validation_scores.index.name = "Number of clusters"
validation_scores = validation_scores[
    ["Sample-wise F1", "Macro F1", "Safety", "Funding", "Efficacy", "Recruitment", "Other"]
]
# Percentage of all training trials still unlabeled after recovery.
reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
validation_scores["Percent unlabeled"] = pd.Series({
    n_clusters: (
        run["part1_train_augmented"][reason_columns].eq(-100).all(axis=1).mean() * 100
    )
    for n_clusters, run in pipeline_results.items()
})
validation_scores["Delta vs baseline"] = (
    validation_scores["Sample-wise F1"] - baseline_validation_metrics["Sample-wise F1"]
)
print(f"Observed-label-only baseline sample-wise F1: {baseline_validation_metrics['Sample-wise F1']:.4f}")
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(validation_scores.round(4))

# Select using the original scores, not their rounded display values.
samplewise_f1_scores = validation_scores["Sample-wise F1"]
best_samplewise_f1 = samplewise_f1_scores.max()
best_cluster_counts = samplewise_f1_scores.index[
    samplewise_f1_scores == best_samplewise_f1
].tolist()
print("Best cluster count(s):", best_cluster_counts)
print(f"Highest validation sample-wise F1: {best_samplewise_f1:.4f}")
# Selecting among 29 settings on 78 validation trials can overfit validation.
# Compare the selected result with the supervised baseline in the report.


In [459]:
def export_part1_predictions(model, X_test, test_df, output_path):
    """Save Kaggle ID,Predicted rows with at least one reason per trial."""
    reason_columns = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
    if len(X_test) != len(test_df):
        raise ValueError("Test embeddings and trial rows must have matching lengths and order.")
    predictions = predict_reason_labels(model, X_test)
    if predictions.shape != (len(test_df), len(reason_columns)):
        raise ValueError("Expected one five-label prediction vector per test trial.")
    if not np.isin(predictions, [0, 1]).all():
        raise ValueError("Reason predictions must be binary.")
    if test_df["ID"].isna().any() or not test_df["ID"].is_unique:
        raise ValueError("Test trial IDs must be present and unique.")

    submission = test_df[["ID"]].copy()
    submission["Predicted"] = [
        " ".join(reason for reason, value in zip(reason_columns, row) if value == 1)
        for row in predictions
    ]
    submission.to_csv(output_path, index=False)
    return submission


In [ ]:
# Use the selected K=21 classifier; its inputs are the original scaled embeddings.
# Kernel PCA was used only to recover labels during training.
selected_run = pipeline_results[21]
selected_reason_model = selected_run["reason_model"]
submission_path = ATTEMPT_DIR / "part1_kmeans_kernel_pca_submission.csv"
part1_submission = export_part1_predictions(
    selected_reason_model, X_test, part1_test, submission_path
)
print("Saved:", submission_path)
print("Test trials:", len(part1_submission))
print("Trials with no predicted reasons:", part1_submission["Predicted"].eq("").sum())
print("Highest-probability fallback trials:", (~selected_reason_model.predict(X_test).any(axis=1)).sum())
part1_submission.head()
